# 📡 Telco Customer Churn – Business Intelligence Dashboard

**Single-notebook pipeline:**
- Data cleaning & EDA
- Churn prediction model (Random Forest + Logistic Regression)
- Dash dashboard with 3 sections:
  - **(a)** Executive Overview
  - **(b)** Customer & Segment Analysis
  - **(c)** Risk, Opportunity & Action

## 0. Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, accuracy_score
)
import dash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

## 1. Load Data

In [ ]:
CSV_PATH = "telco_customer_churn.csv"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"\n[ERROR] Dataset not found at '{CSV_PATH}'.\n"
        "Please download 'WA_Fn-UseC_-Telco-Customer-Churn.csv' from Kaggle:\n"
        "  https://www.kaggle.com/datasets/blastchar/telco-customer-churn\n"
        f"and place it in the same directory as this notebook, renamed to '{CSV_PATH}'."
    )

raw_df = pd.read_csv(CSV_PATH)
print(f"[INFO] Loaded dataset: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
raw_df.head()

## 2. Data Cleaning

In [ ]:
df = raw_df.copy()

# TotalCharges is sometimes loaded as object due to blank strings
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# New customers (tenure=0) may have blank TotalCharges → fill with 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Drop customerID (not predictive)
df.drop(columns=["customerID"], inplace=True, errors="ignore")

# Standardise target
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Strip any leading/trailing whitespace from all object columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

print(f"[INFO] Missing values after cleaning:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\n[INFO] Churn distribution:\n{df['Churn'].value_counts()}")
df.head()

## 3. Feature Engineering

In [ ]:
# Label-encode all remaining categorical columns for the model
categorical_cols = df.select_dtypes(include="object").columns.tolist()
le_dict = {}
df_model = df.copy()
for col in categorical_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    le_dict[col] = le

print(f"Encoded {len(categorical_cols)} categorical columns: {categorical_cols}")

## 4. Churn Prediction Model

In [ ]:
feature_cols = [c for c in df_model.columns if c != "Churn"]
X = df_model[feature_cols]
y = df_model["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale for Logistic Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds  = rf.predict(X_test)
rf_proba  = rf.predict_proba(X_test)[:, 1]
rf_auc    = roc_auc_score(y_test, rf_proba)
rf_acc    = accuracy_score(y_test, rf_preds)

# --- Logistic Regression ---
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_preds  = lr.predict(X_test_sc)
lr_proba  = lr.predict_proba(X_test_sc)[:, 1]
lr_auc    = roc_auc_score(y_test, lr_proba)
lr_acc    = accuracy_score(y_test, lr_preds)

print(f"[MODEL] Random Forest   → Accuracy: {rf_acc:.3f}  AUC: {rf_auc:.3f}")
print(f"[MODEL] Logistic Regr.  → Accuracy: {lr_acc:.3f}  AUC: {lr_auc:.3f}")

# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(f"\n[MODEL] Top-10 Features (Random Forest):\n{feat_imp.head(10)}")

## 5. KPI Calculations

In [ ]:
overall_churn_rate = df["Churn"].mean() * 100                         # %
avg_tenure         = df["tenure"].mean()                               # months
total_customers    = len(df)
churned_customers  = df["Churn"].sum()

# Monthly Recurring Revenue at Risk
mrr_at_risk = df[df["Churn"] == 1]["MonthlyCharges"].sum()

# Churn rate by Contract
churn_by_contract = (
    df.groupby("Contract")["Churn"]
    .agg(["sum", "count"])
    .assign(churn_rate=lambda x: x["sum"] / x["count"] * 100)
    .reset_index()
    .rename(columns={"sum": "churned", "count": "total", "Contract": "Segment"})
)

# Churn rate by PaymentMethod
churn_by_payment = (
    df.groupby("PaymentMethod")["Churn"]
    .agg(["sum", "count"])
    .assign(churn_rate=lambda x: x["sum"] / x["count"] * 100)
    .reset_index()
    .rename(columns={"sum": "churned", "count": "total", "PaymentMethod": "Segment"})
)

# Churn rate by InternetService
churn_by_internet = (
    df.groupby("InternetService")["Churn"]
    .agg(["sum", "count"])
    .assign(churn_rate=lambda x: x["sum"] / x["count"] * 100)
    .reset_index()
    .rename(columns={"sum": "churned", "count": "total", "InternetService": "Segment"})
)

# Tenure buckets
tenure_bins   = [0, 12, 24, 36, 48, 60, 72]
tenure_labels = ["0-12m", "13-24m", "25-36m", "37-48m", "49-60m", "61-72m"]
df["TenureBucket"] = pd.cut(df["tenure"], bins=tenure_bins, labels=tenure_labels, right=True)
churn_by_tenure = (
    df.groupby("TenureBucket", observed=True)["Churn"]
    .agg(["sum", "count"])
    .assign(churn_rate=lambda x: x["sum"] / x["count"] * 100)
    .reset_index()
)

# Monthly charges distribution
monthly_charges_churn = df[["MonthlyCharges", "Churn"]].copy()
monthly_charges_churn["Status"] = monthly_charges_churn["Churn"].map({1: "Churned", 0: "Retained"})

# Top-10 feature importance
top10_features = feat_imp.head(10).reset_index()
top10_features.columns = ["Feature", "Importance"]

# Highest-risk segment
highest_risk_contract = churn_by_contract.loc[churn_by_contract["churn_rate"].idxmax()]
highest_risk_payment  = churn_by_payment.loc[churn_by_payment["churn_rate"].idxmax()]

# MRR breakdown by Contract
mrr_by_contract = (
    df[df["Churn"] == 1]
    .groupby("Contract")["MonthlyCharges"]
    .sum()
    .reset_index()
    .rename(columns={"MonthlyCharges": "MRR_at_Risk"})
)

# Model comparison table
model_comparison = pd.DataFrame({
    "Model": ["Random Forest", "Logistic Regression"],
    "Accuracy": [f"{rf_acc:.2%}", f"{lr_acc:.2%}"],
    "AUC-ROC":  [f"{rf_auc:.3f}", f"{lr_auc:.3f}"],
})

print(f"Overall Churn Rate : {overall_churn_rate:.1f}%")
print(f"MRR at Risk        : ${mrr_at_risk:,.0f}")
print(f"Avg Tenure         : {avg_tenure:.1f} months")
print(f"Total Customers    : {total_customers:,}")
model_comparison

## 6. Colour Palette & Styles

In [ ]:
COLORS = {
    "primary":    "#1a3a5c",
    "accent":     "#e84545",
    "success":    "#27ae60",
    "warning":    "#f39c12",
    "light":      "#f0f4f8",
    "card_bg":    "#ffffff",
    "text":       "#2c3e50",
    "muted":      "#7f8c8d",
    "border":     "#dde3ea",
    "churned":    "#e84545",
    "retained":   "#27ae60",
    "gradient_1": "#1a3a5c",
    "gradient_2": "#2980b9",
}

CARD_STYLE = {
    "background": COLORS["card_bg"],
    "borderRadius": "10px",
    "padding": "20px",
    "boxShadow": "0 2px 8px rgba(0,0,0,0.08)",
    "border": f"1px solid {COLORS['border']}",
    "marginBottom": "16px",
}

SECTION_HEADER = {
    "background": f"linear-gradient(135deg, {COLORS['primary']}, {COLORS['gradient_2']})",
    "color": "#fff",
    "padding": "14px 24px",
    "borderRadius": "8px",
    "marginBottom": "20px",
    "fontSize": "18px",
    "fontWeight": "600",
    "letterSpacing": "0.5px",
}

KPI_CARD = {
    "background": COLORS["card_bg"],
    "borderRadius": "10px",
    "padding": "20px 16px",
    "textAlign": "center",
    "boxShadow": "0 2px 10px rgba(0,0,0,0.07)",
    "border": f"1px solid {COLORS['border']}",
    "flex": "1",
    "minWidth": "160px",
}

## 7. Plotly Figures

In [ ]:
def fig_churn_pie():
    labels = ["Retained", "Churned"]
    values = [total_customers - churned_customers, churned_customers]
    fig = go.Figure(go.Pie(
        labels=labels, values=values,
        hole=0.55,
        marker_colors=[COLORS["retained"], COLORS["churned"]],
        textinfo="label+percent",
        hovertemplate="%{label}: %{value:,}<extra></extra>",
    ))
    fig.update_layout(
        margin=dict(t=20, b=10, l=10, r=10),
        showlegend=False,
        paper_bgcolor="white",
        plot_bgcolor="white",
        height=260,
    )
    return fig


def fig_churn_by_contract():
    fig = px.bar(
        churn_by_contract, x="Segment", y="churn_rate",
        color="Segment",
        color_discrete_sequence=[COLORS["primary"], COLORS["gradient_2"], COLORS["accent"]],
        text=churn_by_contract["churn_rate"].apply(lambda v: f"{v:.1f}%"),
        labels={"churn_rate": "Churn Rate (%)", "Segment": "Contract Type"},
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(
        margin=dict(t=20, b=20, l=20, r=20),
        showlegend=False,
        paper_bgcolor="white", plot_bgcolor="white",
        yaxis_title="Churn Rate (%)", height=300,
        xaxis_title="",
    )
    return fig


def fig_churn_by_payment():
    fig = px.bar(
        churn_by_payment, x="churn_rate", y="Segment",
        orientation="h",
        color="churn_rate",
        color_continuous_scale=["#27ae60", "#f39c12", "#e84545"],
        text=churn_by_payment["churn_rate"].apply(lambda v: f"{v:.1f}%"),
        labels={"churn_rate": "Churn Rate (%)", "Segment": "Payment Method"},
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(
        margin=dict(t=20, b=20, l=20, r=20),
        showlegend=False,
        paper_bgcolor="white", plot_bgcolor="white",
        xaxis_title="Churn Rate (%)", height=300,
        yaxis_title="",
        coloraxis_showscale=False,
    )
    return fig


def fig_churn_by_tenure():
    fig = px.line(
        churn_by_tenure, x="TenureBucket", y="churn_rate",
        markers=True,
        labels={"churn_rate": "Churn Rate (%)", "TenureBucket": "Tenure Group"},
        color_discrete_sequence=[COLORS["accent"]],
    )
    fig.update_traces(line_width=3, marker_size=8)
    fig.update_layout(
        margin=dict(t=20, b=20, l=20, r=20),
        paper_bgcolor="white", plot_bgcolor="white",
        yaxis_title="Churn Rate (%)", height=290,
        xaxis_title="Customer Tenure",
    )
    return fig


def fig_monthly_charges_dist():
    fig = px.histogram(
        monthly_charges_churn, x="MonthlyCharges", color="Status",
        barmode="overlay",
        opacity=0.7,
        color_discrete_map={"Churned": COLORS["churned"], "Retained": COLORS["retained"]},
        labels={"MonthlyCharges": "Monthly Charges ($)", "Status": ""},
        nbins=40,
    )
    fig.update_layout(
        margin=dict(t=20, b=20, l=20, r=20),
        paper_bgcolor="white", plot_bgcolor="white",
        yaxis_title="Customer Count", height=290,
        legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="right", x=1),
    )
    return fig


def fig_feature_importance():
    fig = px.bar(
        top10_features, x="Importance", y="Feature",
        orientation="h",
        color="Importance",
        color_continuous_scale=["#2980b9", "#1a3a5c"],
        labels={"Importance": "Importance Score", "Feature": ""},
    )
    fig.update_layout(
        margin=dict(t=20, b=20, l=20, r=20),
        paper_bgcolor="white", plot_bgcolor="white",
        xaxis_title="Importance Score", height=320,
        coloraxis_showscale=False,
        yaxis=dict(autorange="reversed"),
    )
    return fig


def fig_mrr_at_risk():
    fig = px.bar(
        mrr_by_contract, x="Contract", y="MRR_at_Risk",
        color="Contract",
        color_discrete_sequence=[COLORS["accent"], COLORS["warning"], COLORS["primary"]],
        text=mrr_by_contract["MRR_at_Risk"].apply(lambda v: f"${v:,.0f}"),
        labels={"MRR_at_Risk": "MRR at Risk ($)", "Contract": "Contract Type"},
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(
        margin=dict(t=20, b=20, l=20, r=20),
        showlegend=False,
        paper_bgcolor="white", plot_bgcolor="white",
        yaxis_title="MRR at Risk ($)", height=300,
        xaxis_title="",
    )
    return fig


def fig_internet_churn():
    fig = px.pie(
        churn_by_internet, names="Segment", values="churned",
        color_discrete_sequence=[COLORS["primary"], COLORS["gradient_2"], COLORS["accent"]],
        hole=0.4,
    )
    fig.update_traces(textinfo="label+percent")
    fig.update_layout(
        margin=dict(t=20, b=10, l=10, r=10),
        showlegend=False,
        paper_bgcolor="white",
        height=280,
    )
    return fig


# Preview all figures inline
fig_churn_pie().show()
fig_churn_by_contract().show()
fig_churn_by_payment().show()
fig_churn_by_tenure().show()
fig_monthly_charges_dist().show()
fig_feature_importance().show()
fig_mrr_at_risk().show()
fig_internet_churn().show()

## 8. Dash App Layout

In [ ]:
app = dash.Dash(
    __name__,
    title="Telco Churn Intelligence Dashboard",
    meta_tags=[{"name": "viewport", "content": "width=device-width, initial-scale=1"}],
)

# Helper: KPI card builder
def kpi_card(title, value, subtitle="", color=COLORS["primary"]):
    return html.Div([
        html.P(title, style={
            "fontSize": "12px", "color": COLORS["muted"],
            "marginBottom": "6px", "textTransform": "uppercase", "letterSpacing": "0.8px"
        }),
        html.H3(value, style={
            "fontSize": "28px", "fontWeight": "700",
            "color": color, "margin": "0 0 4px 0"
        }),
        html.P(subtitle, style={"fontSize": "11px", "color": COLORS["muted"], "margin": "0"}),
    ], style=KPI_CARD)


# Helper: section header
def section_header(icon, title, subtitle=""):
    return html.Div([
        html.Span(f"{icon}  {title}", style={"fontWeight": "600", "fontSize": "18px"}),
        html.Br(),
        html.Span(subtitle, style={"fontSize": "12px", "opacity": "0.85"}),
    ], style=SECTION_HEADER)


def graph_card(title, figure, width="100%"):
    return html.Div([
        html.P(title, style={
            "fontWeight": "600", "fontSize": "13px",
            "color": COLORS["text"], "marginBottom": "8px",
            "borderBottom": f"2px solid {COLORS['accent']}",
            "paddingBottom": "6px"
        }),
        dcc.Graph(figure=figure, config={"displayModeBar": False}),
    ], style={**CARD_STYLE, "width": width, "boxSizing": "border-box"})


app.layout = html.Div(style={
    "fontFamily": "-apple-system, 'Segoe UI', Roboto, sans-serif",
    "backgroundColor": COLORS["light"],
    "minHeight": "100vh",
    "padding": "0",
}, children=[

    # ── HEADER BAR ──────────────────────────────────────────────────────────
    html.Div([
        html.Div([
            html.H1("📡 Telco Churn Intelligence", style={
                "margin": "0", "fontSize": "22px", "fontWeight": "700", "color": "#fff"
            }),
            html.P("Customer Churn Prediction & Business Analytics Dashboard", style={
                "margin": "0", "fontSize": "12px", "color": "rgba(255,255,255,0.75)"
            }),
        ]),
        html.Div([
            html.Span(f"Model: Random Forest  |  AUC: {rf_auc:.3f}  |  Accuracy: {rf_acc:.2%}", style={
                "fontSize": "12px", "color": "rgba(255,255,255,0.85)",
                "background": "rgba(255,255,255,0.12)", "padding": "6px 14px",
                "borderRadius": "20px"
            }),
        ]),
    ], style={
        "background": f"linear-gradient(135deg, {COLORS['primary']}, {COLORS['gradient_2']})",
        "padding": "18px 32px",
        "display": "flex",
        "justifyContent": "space-between",
        "alignItems": "center",
    }),

    html.Div(style={"padding": "24px 32px"}, children=[

        # ════════════════════════════════════════════════════════════════════
        # SECTION A: EXECUTIVE OVERVIEW
        # ════════════════════════════════════════════════════════════════════
        section_header("📊", "Executive Overview",
                        "Top-level KPIs and churn landscape at a glance"),

        # KPI Row
        html.Div([
            kpi_card("Overall Churn Rate",
                     f"{overall_churn_rate:.1f}%",
                     f"{churned_customers:,} of {total_customers:,} customers",
                     COLORS["accent"]),
            kpi_card("MRR at Risk",
                     f"${mrr_at_risk:,.0f}",
                     "Monthly revenue from churned customers",
                     COLORS["warning"]),
            kpi_card("Avg. Customer Tenure",
                     f"{avg_tenure:.1f} mo",
                     "Across all customers",
                     COLORS["gradient_2"]),
            kpi_card("Total Customers",
                     f"{total_customers:,}",
                     f"{churned_customers:,} churned",
                     COLORS["primary"]),
            kpi_card("Model AUC (RF)",
                     f"{rf_auc:.3f}",
                     "Random Forest prediction accuracy",
                     COLORS["success"]),
        ], style={
            "display": "flex", "gap": "16px", "flexWrap": "wrap",
            "marginBottom": "20px"
        }),

        # Churn pie + MRR at risk by contract
        html.Div([
            html.Div([
                graph_card("Overall Churn Split", fig_churn_pie(), width="100%"),
            ], style={"flex": "1", "minWidth": "280px"}),
            html.Div([
                graph_card("MRR at Risk by Contract Type", fig_mrr_at_risk(), width="100%"),
            ], style={"flex": "2", "minWidth": "380px"}),
        ], style={"display": "flex", "gap": "16px", "flexWrap": "wrap", "marginBottom": "8px"}),

        # Model comparison table
        html.Div([
            html.P("Model Performance Comparison", style={
                "fontWeight": "600", "fontSize": "13px", "color": COLORS["text"],
                "marginBottom": "10px", "borderBottom": f"2px solid {COLORS['accent']}",
                "paddingBottom": "6px"
            }),
            dash_table.DataTable(
                data=model_comparison.to_dict("records"),
                columns=[{"name": c, "id": c} for c in model_comparison.columns],
                style_table={"overflowX": "auto"},
                style_header={
                    "backgroundColor": COLORS["primary"], "color": "#fff",
                    "fontWeight": "600", "fontSize": "12px",
                },
                style_cell={
                    "textAlign": "center", "padding": "10px 16px",
                    "fontSize": "13px", "border": f"1px solid {COLORS['border']}",
                },
                style_data_conditional=[
                    {"if": {"row_index": 0},
                     "backgroundColor": "#eaf6ee", "color": COLORS["success"], "fontWeight": "700"}
                ],
            ),
        ], style={**CARD_STYLE, "marginBottom": "32px"}),


        # ════════════════════════════════════════════════════════════════════
        # SECTION B: CUSTOMER & SEGMENT ANALYSIS
        # ════════════════════════════════════════════════════════════════════
        section_header("🔍", "Customer & Segment Analysis",
                        "Churn drivers by contract type, tenure, and payment method"),

        html.Div([
            html.Div([
                graph_card("Churn Rate by Contract Type", fig_churn_by_contract(), width="100%"),
            ], style={"flex": "1", "minWidth": "300px"}),
            html.Div([
                graph_card("Churn Rate by Payment Method", fig_churn_by_payment(), width="100%"),
            ], style={"flex": "1", "minWidth": "300px"}),
        ], style={"display": "flex", "gap": "16px", "flexWrap": "wrap", "marginBottom": "8px"}),

        html.Div([
            html.Div([
                graph_card("Churn Rate by Customer Tenure", fig_churn_by_tenure(), width="100%"),
            ], style={"flex": "1", "minWidth": "300px"}),
            html.Div([
                graph_card("Monthly Charges Distribution (Churned vs Retained)",
                           fig_monthly_charges_dist(), width="100%"),
            ], style={"flex": "1", "minWidth": "300px"}),
        ], style={"display": "flex", "gap": "16px", "flexWrap": "wrap", "marginBottom": "8px"}),

        html.Div([
            html.Div([
                graph_card("Churned Customers by Internet Service", fig_internet_churn(), width="100%"),
            ], style={"flex": "1", "minWidth": "300px"}),
            html.Div([
                graph_card("Top-10 Churn Predictors (Random Forest)", fig_feature_importance(), width="100%"),
            ], style={"flex": "2", "minWidth": "400px"}),
        ], style={"display": "flex", "gap": "16px", "flexWrap": "wrap", "marginBottom": "32px"}),


        # ════════════════════════════════════════════════════════════════════
        # SECTION C: RISK, OPPORTUNITY & ACTION
        # ════════════════════════════════════════════════════════════════════
        section_header("🚨", "Risk, Opportunity & Action",
                        "Highest-risk segments, growth levers, and recommended retention actions"),

        html.Div([

            # Risk card
            html.Div([
                html.Div("🔴  Highest-Risk Segment", style={
                    "fontWeight": "700", "fontSize": "14px",
                    "color": COLORS["accent"], "marginBottom": "10px"
                }),
                html.P([
                    html.Strong("Contract Type: "),
                    html.Span(
                        f"{highest_risk_contract['Segment']}  →  "
                        f"{highest_risk_contract['churn_rate']:.1f}% churn rate",
                        style={"color": COLORS["accent"], "fontWeight": "600"}
                    ),
                ], style={"fontSize": "14px", "margin": "4px 0"}),
                html.P([
                    html.Strong("Payment Method: "),
                    html.Span(
                        f"{highest_risk_payment['Segment']}  →  "
                        f"{highest_risk_payment['churn_rate']:.1f}% churn rate",
                        style={"color": COLORS["warning"], "fontWeight": "600"}
                    ),
                ], style={"fontSize": "14px", "margin": "4px 0"}),
                html.Hr(style={"borderColor": COLORS["border"]}),
                html.P(
                    "Month-to-month contract holders paying via Electronic Check show the "
                    "highest churn propensity. These customers are typically newer (0–12 months) "
                    "and have not yet built a loyalty anchor with the provider.",
                    style={"fontSize": "13px", "color": COLORS["text"], "lineHeight": "1.6"}
                ),
            ], style={**CARD_STYLE, "flex": "1", "minWidth": "280px",
                      "borderLeft": f"4px solid {COLORS['accent']}"}),

            # Opportunity card
            html.Div([
                html.Div("🟢  Growth Opportunity", style={
                    "fontWeight": "700", "fontSize": "14px",
                    "color": COLORS["success"], "marginBottom": "10px"
                }),
                html.P([
                    html.Strong("Two-Year Contract Customers: "),
                    html.Span(
                        f"Only {churn_by_contract[churn_by_contract['Segment'] == 'Two year']['churn_rate'].values[0]:.1f}% churn rate",
                        style={"color": COLORS["success"], "fontWeight": "600"}
                    ),
                ], style={"fontSize": "14px", "margin": "4px 0"}),
                html.Hr(style={"borderColor": COLORS["border"]}),
                html.P(
                    "Customers on two-year contracts have dramatically lower churn. "
                    "The largest growth opportunity lies in migrating month-to-month customers "
                    "to long-term contracts through targeted incentives (discounts, free upgrades, "
                    "or loyalty rewards). Even a 10% migration could save significant MRR.",
                    style={"fontSize": "13px", "color": COLORS["text"], "lineHeight": "1.6"}
                ),
                html.P(
                    f"💡 Potential MRR saved if 10% of M2M customers are retained: "
                    f"${mrr_by_contract[mrr_by_contract['Contract'] == 'Month-to-month']['MRR_at_Risk'].values[0] * 0.10:,.0f}/mo"
                    if 'Month-to-month' in mrr_by_contract['Contract'].values else "",
                    style={"fontSize": "13px", "color": COLORS["success"], "fontWeight": "600"}
                ),
            ], style={**CARD_STYLE, "flex": "1", "minWidth": "280px",
                      "borderLeft": f"4px solid {COLORS['success']}"}),

            # Action card
            html.Div([
                html.Div("⚡  Recommended Retention Actions", style={
                    "fontWeight": "700", "fontSize": "14px",
                    "color": COLORS["gradient_2"], "marginBottom": "10px"
                }),
                html.Ul([
                    html.Li("Launch a Contract Upgrade Campaign: Offer month-to-month customers a 15% discount to switch to a 1- or 2-year plan within the next 30 days.", style={"marginBottom": "8px"}),
                    html.Li("Electronic Check Intervention: Customers using electronic check churn at the highest rate. Introduce an auto-pay incentive (e.g., $5/mo credit) to shift to bank transfer or credit card.", style={"marginBottom": "8px"}),
                    html.Li("Early Tenure Nurture Program: Deploy a 90-day onboarding journey for new customers (tenure < 12 months) with proactive support check-ins and feature education.", style={"marginBottom": "8px"}),
                    html.Li("Fibre-Optic Satisfaction Audit: Fibre customers show higher churn — investigate service quality, pricing fairness, and competitor pressure in this segment.", style={"marginBottom": "8px"}),
                    html.Li("AI-Powered Churn Alerts: Deploy the trained Random Forest model in production to flag high-risk customers weekly and trigger automated retention outreach.", style={"marginBottom": "8px"}),
                ], style={"fontSize": "13px", "color": COLORS["text"], "lineHeight": "1.7",
                          "paddingLeft": "18px"}),
            ], style={**CARD_STYLE, "flex": "2", "minWidth": "340px",
                      "borderLeft": f"4px solid {COLORS['gradient_2']}"}),

        ], style={"display": "flex", "gap": "16px", "flexWrap": "wrap", "marginBottom": "32px"}),

        # Footer
        html.Div([
            html.P(
                "Telco Customer Churn BI Dashboard  •  Built with Python, Dash & Plotly  "
                f"•  Random Forest AUC: {rf_auc:.3f}  •  Dataset: WA_Fn-UseC_-Telco-Customer-Churn",
                style={"fontSize": "11px", "color": COLORS["muted"], "textAlign": "center", "margin": "0"}
            ),
        ], style={
            "borderTop": f"1px solid {COLORS['border']}",
            "paddingTop": "14px", "marginTop": "8px"
        }),

    ]),  # end main padding div
])

## 9. Run the Dashboard

Execute the cell below to launch the Dash server.  
Then open **http://127.0.0.1:8050** in your browser.

In [ ]:
if __name__ == "__main__":
    print("\n" + "="*60)
    print("  Telco Churn BI Dashboard")
    print(f"  Open: http://127.0.0.1:8050")
    print("="*60 + "\n")
    app.run(debug=False, host="127.0.0.1", port=8050)